# Conclusiones: qué ha rendido más

Este notebook lee los resultados de los otros tres análisis. Si faltan o cambia
la fecha, el catálogo, el código, la configuración o un CSV manual, recalcula el pipeline necesario.
También puedes actualizarlo todo con `tr-pipeline notebooks`.

**Lectura rápida:** el puesto 1 es el activo que más ha subido en el último año, en euros.
Es una clasificación de rendimiento pasado. No indica qué comprar.
Un informe parcial se identifica antes de mostrar sus rankings.


In [1]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["TR_PIPELINE_ROOT"]).resolve() if os.getenv("TR_PIPELINE_ROOT") else Path.cwd()
if not (ROOT / "config.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from tr_pipeline.pipeline import run_asset_type, conclusions
from tr_pipeline.notebook import show_coverage, show_eda, show_rankings
from IPython.display import display, Markdown
import pandas as pd

REFRESH = False  # True fuerza una descarga nueva de Yahoo y divisas.
OFFLINE = os.getenv("TR_PIPELINE_OFFLINE", "0") == "1"


In [2]:
RESULTS = conclusions(ROOT, refresh_dependencies=True, offline=OFFLINE)

In [3]:
for kind, result in RESULTS.items():
    show_coverage(result)
    show_rankings(result)
    if kind == "stock":
        metrics = pd.read_csv(ROOT / "data/processed/stock/metrics.csv")
        if "pe_discount_pct" in metrics:
            candidates = metrics.loc[metrics.pe_discount_pct.notna()].sort_values("pe_discount_pct", ascending=False).head(5)
            display(Markdown("### PER: candidatos para revisar"))
            if len(candidates):
                display(candidates[["isin", "name", "pe_current", "pe_historical_median", "valuation_signal"]])
            else:
                display(Markdown("No hay comparaciones históricas suficientes y fiables."))

### Acciones
Referencia: **2026-11-04**. Catálogo cargado: **250** ISIN. Con precios: **188**. Elegibles: **184**. Pendientes/error: **62**.

Selección: **250** de **11323** publicados. Criterio: Orden predeterminado del explorador público TR (proxy de relevancia; fórmula no publicada). El ranking por rentabilidad se calcula después sobre esta selección.

**RESULTADOS PARCIALES: no son el top de todo Trade Republic.**

Selección: 250 ISIN de 11323 publicados; catálogo cargado: 250. Criterio: Orden predeterminado del explorador público TR (proxy de relevancia; fórmula no publicada). La disponibilidad en una cuenta concreta puede variar.

### Top 5 global por rentabilidad a un año en EUR

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | US5951121038 | Micron Technology | 272.41% |
| 2 | NO0011082075 | Höegh Autoliners | 153.83% |
| 3 | US24703L2025 | Dell Technologies | 148.17% |
| 4 | US7960508882 | Samsung (GDR) | 127.77% |
| 5 | DE0005936124 | OHB | 105.96% |

### Top 5 por sector o categoría

**Alcohol & Tobacco**

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | US02209S1033 | Altria Group | 48.21% |
| 2 | GB0002875804 | British American Tobacco | 27.91% |
| 3 | GB0002374006 | Diageo | 9.47% |
| 4 | GB0004544929 | Imperial Brands | 7.86% |

... (100 outputs left out)

**Real Estate**

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | IE00B1FZS350 | Developed Markets Property Yield USD (Dist) | 9.41% |

**Stocks (sin sector específico)**

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | IE00BHZRR030 | FTSE Korea USD (Acc) | 73.10% |
| 2 | IE000I8KRLL9 | MSCI Global Semiconductors USD (Acc) | 58.42% |
| 3 | IE00BYPLS672 | Cyber Security USD (Acc) | 49.59% |
| 4 | IE00082MOBL9 | European SuperDividend EUR (Dis) | 46.39% |
| 5 | IE00BLPK3577 | Cybersecurity USD (Acc) | 43.18% |

**Telecommunication**

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | IE00B3VSSL01 | Technology S&P US Select Sector USD (Acc) | 18.08% |
| 2 | LU1681038243 | NASDAQ 100 EUR (Acc) | 14.20% |

**Vehicles**

| Puesto | ISIN | Nombre | Rentabilidad 1 año |
| --- | --- | --- | ---: |
| 1 | IE00BGL86Z12 | Electric Driving Tech USD (Acc) | 24.68% |

Se ha guardado la misma lectura en `reports/conclusiones.md`, junto con
los CSV de top 5 y un manifest que identifica los análisis de origen.
Consulta `data/processed/<tipo>/failures.csv` para ver los ISIN pendientes de datos.
